# 5. 图像分割

**目标：从图像中分离目标区域，掌握传统分割方法。**

本 Notebook 系统讲解 OpenCV 中阈值处理与图像分割的核心方法。

主要内容：
1. 固定阈值处理
2. OTSU 阈值算法
3. 多阈值处理算法
4. 自适应阈值处理
5. 移动平均阈值处理
6. HSV 颜色空间的阈值分割
7. 区域生长与分离
8. 区域分离与聚合
9. 超像素区域分割：SLIC、SEEDS、LSC
10. 分水岭算法
11. 图割分割算法：GraphCut、GrabCut
12. 均值漂移算法
13. 运动图像分割：帧间差分法、背景差分法、密集光流法（与模块 11 衔接）

**独立性：依赖模块 1、3、4、6，是传统视觉核心模块。**

# 1. 固定阈值处理

## 1.1 固定阈值的基本原理

固定阈值是最简单的图像分割方法。
它的核心思想是：给定一个阈值 `thresh`，将图像中每个像素的灰度值与 `thresh` 比较，
根据比较结果将像素分为两类（通常为 0 和 255）。

数学表达：

```
dst(x, y) = maxVal,  if src(x, y) > thresh
dst(x, y) = 0,       otherwise
```

OpenCV 提供了 `cv2.threshold()` 函数来实现多种阈值处理。

## 1.2 `cv2.threshold()` 函数详解

```python
retval, dst = cv2.threshold(src, thresh, maxval, type)
```

参数说明：
- `src`：输入图像，通常为单通道灰度图。
- `thresh`：阈值。
- `maxval`：当像素值超过（或小于，取决于 type）阈值时赋予的新值。
- `type`：阈值处理类型，常见的有：
  - `cv2.THRESH_BINARY`：超过阈值取 maxval，否则取 0。
  - `cv2.THRESH_BINARY_INV`：超过阈值取 0，否则取 maxval。
  - `cv2.THRESH_TRUNC`：超过阈值取阈值，否则保持原值。
  - `cv2.THRESH_TOZERO`：超过阈值保持原值，否则取 0。
  - `cv2.THRESH_TOZERO_INV`：超过阈值取 0，否则保持原值。

返回值：
- `retval`：实际使用的阈值（在使用 OTSU 等自动阈值时会返回计算出的阈值）。
- `dst`：阈值处理后的图像。

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img = cv2.imread('p&v/grayLena.bmp', cv2.IMREAD_GRAYSCALE)       # 以灰度模式读取图像，返回单通道数组
if img is None:                                                  # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

ret, thresh_binary = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)  # 固定阈值二值化：大于 127 变 255，否则变 0
print('实际使用的阈值:', ret)                                    # 打印实际使用的阈值

plt.figure(figsize=(10, 5))                                      # 创建画布
plt.subplot(1, 2, 1)                                             # 第 1 个子图
plt.imshow(img, cmap='gray')                                     # 显示原始灰度图
plt.title('Original Gray')                                       # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 2, 2)                                             # 第 2 个子图
plt.imshow(thresh_binary, cmap='gray')                           # 显示二值化结果
plt.title('THRESH_BINARY (thresh=127)')                          # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

## 1.3 五种固定阈值类型对比

下面用同一张灰度图，对比五种阈值类型的处理效果。

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img = cv2.imread('p&v/grayLena.bmp', cv2.IMREAD_GRAYSCALE)       # 以灰度模式读取图像
if img is None:                                                  # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

ret, th1 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)      # 二值化：大于阈值取 255，否则取 0
ret, th2 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY_INV)  # 反二值化：大于阈值取 0，否则取 255
ret, th3 = cv2.threshold(img, 127, 255, cv2.THRESH_TRUNC)       # 截断：大于阈值取阈值，否则保持原值
ret, th4 = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO)      # 置零：大于阈值保持原值，否则取 0
ret, th5 = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO_INV)  # 反置零：大于阈值取 0，否则保持原值

titles = ['Original', 'BINARY', 'BINARY_INV', 'TRUNC', 'TOZERO', 'TOZERO_INV']  # 子图标题列表
images = [img, th1, th2, th3, th4, th5]                          # 图像列表

plt.figure(figsize=(12, 8))                                      # 创建画布
for i in range(6):                                               # 遍历 6 张图
    plt.subplot(2, 3, i + 1)                                     # 创建 2 行 3 列子图
    plt.imshow(images[i], cmap='gray')                           # 以灰度色图显示
    plt.title(titles[i])                                         # 设置标题
    plt.axis('off')                                              # 关闭坐标轴
plt.tight_layout()                                               # 自动调整布局
plt.show()                                                       # 显示图像

# 2. OTSU 阈值算法

## 2.1 OTSU 原理

OTSU（大津法）是一种自动确定阈值的算法，适用于双峰直方图的图像。
它通过最大化类间方差来寻找最佳阈值。

在 OpenCV 中，只需在 `cv2.threshold()` 的 `type` 参数中加上 `cv2.THRESH_OTSU` 即可。
此时 `thresh` 参数会被忽略，函数返回实际计算出的阈值。

下面用一张经典的 `grayLena.bmp` 来演示 OTSU 的完整过程，并绘制直方图与类间方差曲线，直观展示它是如何找到阈值的。

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img = cv2.imread('p&v/grayLena.bmp', cv2.IMREAD_GRAYSCALE)       # 以灰度模式读取图像
if img is None:                                                  # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

ret_otsu, th_otsu = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)  # OTSU 自动阈值二值化
print('OTSU 计算出的阈值:', ret_otsu)                           # 打印 OTSU 计算出的阈值

# 计算灰度直方图：统计每个灰度级（0~255）出现的像素个数
hist = cv2.calcHist([img], [0], None, [256], [0, 256]).ravel()   # 返回长度为 256 的一维数组

# 计算 OTSU 的类间方差曲线：遍历每个候选阈值，计算前景/背景两类的类间方差
total = img.size                                                 # 图像总像素数
sum_total = np.dot(np.arange(256), hist)                         # 全图灰度总和，用于计算均值
w_bg = np.cumsum(hist)                                           # 背景累计像素数（灰度 <= t 的部分）
w_fg = total - w_bg                                              # 前景累计像素数（灰度 > t 的部分）
sum_bg = np.cumsum(np.arange(256) * hist)                        # 背景累计灰度和
sum_fg = sum_total - sum_bg                                      # 前景累计灰度和

# 避免除零：将权重为 0 的位置替换为 1（这些位置的方差会被置 0，不影响最终结果）
w_bg_safe = np.where(w_bg == 0, 1, w_bg)                         # 背景权重安全化
w_fg_safe = np.where(w_fg == 0, 1, w_fg)                         # 前景权重安全化

mean_bg = sum_bg / w_bg_safe                                     # 背景平均灰度
mean_fg = sum_fg / w_fg_safe                                     # 前景平均灰度
var_between = w_bg * w_fg * (mean_bg - mean_fg) ** 2             # 类间方差公式
var_between[(w_bg == 0) | (w_fg == 0)] = 0                       # 无效位置的类间方差置 0

plt.figure(figsize=(15, 10))                                     # 创建画布

plt.subplot(2, 2, 1)                                             # 第 1 个子图
plt.imshow(img, cmap='gray')                                     # 显示原始灰度图
plt.title('Original Gray')                                       # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(2, 2, 2)                                             # 第 2 个子图
plt.imshow(th_otsu, cmap='gray')                                 # 显示 OTSU 二值化结果
plt.title(f'OTSU (thresh={ret_otsu})')                           # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(2, 2, 3)                                             # 第 3 个子图：灰度直方图
plt.plot(hist, color='black')                                    # 绘制灰度直方图曲线
plt.axvline(ret_otsu, color='red', linestyle='--',               # 用红色虚线标出 OTSU 阈值
            label=f'OTSU thresh={ret_otsu}')                     # 设置图例文字
plt.title('Grayscale Histogram')                                 # 设置标题
plt.xlabel('Pixel Value')                                        # 横轴：像素灰度值
plt.ylabel('Count')                                              # 纵轴：像素个数
plt.legend()                                                     # 显示图例

plt.subplot(2, 2, 4)                                             # 第 4 个子图：类间方差曲线
plt.plot(var_between, color='blue')                              # 绘制类间方差随阈值变化的曲线
plt.axvline(ret_otsu, color='red', linestyle='--',               # 用红色虚线标出 OTSU 阈值
            label=f'OTSU thresh={ret_otsu}')                     # 设置图例文字
plt.title('Between-class Variance vs Threshold')                 # 设置标题
plt.xlabel('Threshold')                                          # 横轴：候选阈值
plt.ylabel('Between-class Variance')                             # 纵轴：类间方差
plt.legend()                                                     # 显示图例

plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

**直方图与方差曲线解读：**

- **左图（灰度直方图）**：`grayLena.bmp` 呈现非常典型的**双峰分布**。左侧小峰对应背景（暗部），右侧大峰对应人脸和肩膀（亮部）。两峰之间的山谷最低点，就是理想的阈值位置。红色虚线标出的 OTSU 阈值正好落在这个低谷附近。
- **右图（类间方差曲线）**：曲线展示了“如果选用不同阈值，前景和背景的类间方差会如何变化”。在阈值较低时方差很小，随着阈值增加方差逐渐上升，在 OTSU 阈值处达到**最高点（峰值）**，之后又下降。OTSU 算法本质上就是在这个曲线上寻找最大值。

这两张图完美配合，解释了 OTSU 的数学本质：**在双峰直方图的山谷处，类间方差达到最大，分割效果最好。**

# 3. 多阈值处理算法

## 3.1 多阈值原理

当图像有多个目标区域时，单一阈值可能无法很好分割。
多阈值处理通过设定多个阈值，将图像分为多个区域。

OpenCV 没有直接的多阈值函数，但可以通过多次调用 `cv2.threshold()` 并组合结果来实现。

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img = cv2.imread('p&v/grayLena.bmp', cv2.IMREAD_GRAYSCALE)       # 以灰度模式读取图像
if img is None:                                                  # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

ret1, th_low = cv2.threshold(img, 80, 255, cv2.THRESH_BINARY)    # 低阈值二值化：大于 80 变 255
ret2, th_high = cv2.threshold(img, 160, 255, cv2.THRESH_BINARY)  # 高阈值二值化：大于 160 变 255

multi_thresh = np.zeros_like(img)                                # 创建全零图像
multi_thresh[th_low > 0] = 100                                   # 低阈值区域赋值为 100（中等亮度）
multi_thresh[th_high > 0] = 200                                  # 高阈值区域赋值为 200（高亮部）

plt.figure(figsize=(10, 5))                                      # 创建画布
plt.subplot(1, 2, 1)                                             # 第 1 个子图
plt.imshow(img, cmap='gray')                                     # 显示原始灰度图
plt.title('Original Gray')                                       # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 2, 2)                                             # 第 2 个子图
plt.imshow(multi_thresh, cmap='gray')                            # 显示多阈值结果
plt.title('Multi-threshold (80, 160)')                           # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

**多阈值分割效果解读：**

从上面的结果可以看到，原本连续的 256 个灰度级被压缩成了 3 个离散的灰度级：

- **黑色（0）**：原始灰度 < 80 的极暗区域，对应头发阴影、帽子内部、背景最暗处。
- **深灰色（100）**：80 ≤ 原始灰度 < 160 的中等亮度区域，对应背景墙壁、帽子侧面、头发中较亮的部分。
- **浅灰色（200）**：原始灰度 ≥ 160 的明亮区域，对应 Lena 的面部、肩膀、帽子顶部等受光面。

**现象与特点：**

- **色块化/海报化**：原本平滑的灰度过渡变成了生硬的色块边界，细节（如脸颊阴影、头发光泽）丢失。
- **噪声放大**：在灰度值接近 80 或 160 的区域，会出现细碎的噪点或“椒盐”效果。
- **区域划分明确**：适合快速提取特定亮度范围的目标。例如，如果想提取 Lena 的脸部，只需关注浅灰色区域即可。

**适用场景与局限：**

- **适用**：目标与背景亮度差异明显、只需要粗略区域划分的场景。
- **局限**：对于灰度变化丰富的自然图像，会丢失大量细节，不适合需要保留平滑过渡的任务。
- **改进方向**：可以改用自适应阈值、OTSU 自动阈值、K-means 聚类等方法，或在阈值分割后做形态学开闭运算来平滑边界。

# 4. 自适应阈值处理

## 4.1 自适应阈值原理

固定阈值对光照不均匀的图像效果很差。
自适应阈值根据每个像素邻域的统计信息（均值或高斯加权和）计算局部阈值。

```python
dst = cv2.adaptiveThreshold(src, maxValue, adaptiveMethod, thresholdType, blockSize, C)
```

参数说明：
- `src`：输入灰度图。
- `maxValue`：满足条件时赋予的最大值。
- `adaptiveMethod`：
  - `cv2.ADAPTIVE_THRESH_MEAN_C`：邻域均值。
  - `cv2.ADAPTIVE_THRESH_GAUSSIAN_C`：邻域高斯加权和。
- `thresholdType`：`cv2.THRESH_BINARY` 或 `cv2.THRESH_BINARY_INV`。
- `blockSize`：邻域大小，必须为奇数。
- `C`：从计算出的阈值中减去的常数。

### 4.1.1 固定阈值的局限

在固定阈值中，我们只有一个全局阈值 \( T \)（比如 127）。
对于图像中的每一个像素 \((x, y)\)，判断规则是：

```
dst(x, y) = 255,  if src(x, y) > T
dst(x, y) = 0,    otherwise
```

**问题**：如果图像左上角很亮（像素值普遍 200），右下角很暗（像素值普遍 50），那么一个全局的 \( T=127 \) 会导致：
- 左上角：几乎所有像素都 > 127，全部变成 255（全白）。
- 右下角：几乎所有像素都 < 127，全部变成 0（全黑）。
- 结果：分割失败，因为光照不均导致同一个阈值无法同时适配明暗区域。

### 4.1.2 自适应阈值的核心思想

自适应阈值不再使用一个全局的 \( T \)，而是**为每个像素 \((x, y)\) 单独计算一个局部阈值 \( T(x, y) \)**。

这个 \( T(x, y) \) 是基于该像素**周围邻域**（比如 11×11 的窗口）内的像素统计信息计算出来的。

判断规则变为：

```
dst(x, y) = 255,  if src(x, y) > T(x, y)
dst(x, y) = 0,    otherwise
```

**关键点**：\( T(x, y) \) 是随位置变化的，所以叫“自适应”。

### 4.1.3 \( T(x, y) \) 的两种计算方法

OpenCV 提供了两种计算 \( T(x, y) \) 的方法：

**方法一：均值法（`cv2.ADAPTIVE_THRESH_MEAN_C`）**

```
T(x, y) = mean(邻域(x, y)) - C
```

- **邻域**：以 \((x, y)\) 为中心，大小为 `blockSize × blockSize` 的矩形窗口（比如 11×11）。
- **mean**：该窗口内所有像素的**算术平均值**。
- **C**：一个常数，用来微调阈值。通常取正数，让阈值略低于均值，从而保留更多前景。

**举例**：
假设 `blockSize=11`，`C=2`，当前像素 \((x, y)\) 的 11×11 邻域内像素平均值为 150。
则 \( T(x, y) = 150 - 2 = 148 \)。
如果 \( src(x, y) = 160 > 148 \)，则该像素被设为 255；否则设为 0。

**方法二：高斯加权法（`cv2.ADAPTIVE_THRESH_GAUSSIAN_C`）**

```
T(x, y) = gaussian_weighted_sum(邻域(x, y)) - C
```

- **gaussian_weighted_sum**：不是简单平均，而是用**高斯核**对邻域内像素加权求和。
- 高斯核的特点是：**离中心像素越近的像素权重越大，越远的权重越小**。
- 这样计算出的 \( T(x, y) \) 更能反映中心像素周围的局部特性，对噪声更鲁棒。

**举例**：
同样 `blockSize=11`，`C=2`，邻域内像素经过高斯加权求和后得到 152。
则 \( T(x, y) = 152 - 2 = 150 \)。

### 4.1.4 为什么这样就能“自适应”？

因为 \( T(x, y) \) 是**逐像素、逐邻域**计算的：

- 在**亮区**：邻域均值高，\( T(x, y) \) 也高，只有比周围更亮的像素才会被保留。
- 在**暗区**：邻域均值低，\( T(x, y) \) 也低，同样只有比周围更亮的像素才会被保留。
- 在**边缘**：邻域内既有亮像素也有暗像素，均值处于中间，能较好地分离出边缘。

这就相当于给每个像素都配了一个“局部尺子”，而不是用一把“全局尺子”去量整张图。

**用 Lena 图直观理解：**

- **左脸**（较亮）：邻域均值约 180，\( T \approx 178 \)。像素值 190 的被保留，像素值 170 的被置 0。
- **头发**（较暗）：邻域均值约 60，\( T \approx 58 \)。像素值 70 的被保留，像素值 50 的被置 0。
- **背景墙壁**（中等）：邻域均值约 120，\( T \approx 118 \)。

这样，无论亮区还是暗区，都能根据**局部对比度**做出合理判断，而不是被全局阈值一刀切。

### 4.1.5 参数影响

| 参数 | 作用 | 典型值 | 影响 |
|------|------|--------|------|
| `blockSize` | 邻域大小 | 11, 15, 21 | 太小：对噪声敏感；太大：失去局部性，接近全局阈值 |
| `C` | 阈值偏移量 | 2, 5, 10 | 越大：阈值越低，保留更多前景；越小：保留更少 |
| `adaptiveMethod` | 计算方法 | MEAN 或 GAUSSIAN | MEAN 更简单；GAUSSIAN 对噪声更鲁棒 |

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img = cv2.imread('p&v/grayLena.bmp', cv2.IMREAD_GRAYSCALE)       # 以灰度模式读取图像
if img is None:                                                  # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

th_adapt_mean = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_MEAN_C,     # 自适应均值阈值
                                      cv2.THRESH_BINARY, 11, 2)                 # 邻域 11x11，常数 C=2
th_adapt_gauss = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, # 自适应高斯阈值
                                       cv2.THRESH_BINARY, 11, 2)                # 邻域 11x11，常数 C=2

plt.figure(figsize=(15, 5))                                      # 创建画布
plt.subplot(1, 3, 1)                                             # 第 1 个子图
plt.imshow(img, cmap='gray')                                     # 显示原始灰度图
plt.title('Original Gray')                                       # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 3, 2)                                             # 第 2 个子图
plt.imshow(th_adapt_mean, cmap='gray')                           # 显示自适应均值结果
plt.title('Adaptive Mean')                                       # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 3, 3)                                             # 第 3 个子图
plt.imshow(th_adapt_gauss, cmap='gray')                          # 显示自适应高斯结果
plt.title('Adaptive Gaussian')                                   # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

### 4.1.6 自适应阈值处理后的图像有什么用？

你可能会问：自适应阈值处理后，图像从连续灰度变成了不连续的 0/255，
这看起来只是让图像“变粗糙”了，到底有什么用？

**关键理解**：阈值化不是为了“好看”，而是为了**把图像从‘灰度描述’变成‘区域描述’**，
从而为后续的**测量、识别、决策**服务。

#### 从“连续”到“不连续”，到底改变了什么？

| 维度 | 原始灰度图 | 阈值化后的二值图 |
|------|-----------|-----------------|
| 数据形式 | 每个像素 0~255 连续值 | 每个像素只有 0/255 |
| 描述对象 | 亮度 | **区域归属** |
| 计算机能做什么 | 只能比较大小、算梯度 | **可以直接计数、测量面积、判断形状** |
| 人眼看到 | 平滑的明暗过渡 | 生硬的色块 |

关键变化是：**图像从“像素级亮度”变成了“区域级标签”**。

#### 自适应阈值处理后的图像，具体能用来干什么？

1. **计数与测量**
   - 细胞计数：显微镜下细胞核比背景亮，自适应阈值后，白色连通区域的数量就是细胞数。
   - PCB 焊点检测：焊点反光强，阈值后白色区域面积可以判断焊点是否饱满。
   - 颗粒分析：矿石、药片、种子等，阈值后统计颗粒个数、粒径分布。

2. **缺陷检测**
   - 工业表面划痕、裂纹在灰度图里可能只有轻微亮度变化。
   - 自适应阈值后，缺陷变成明显的白色线条或斑块，方便后续定位。

3. **文本与条码提取**
   - 文档扫描件光照不均，固定阈值会把一部分文字吞掉。
   - 自适应阈值后，每个字都能被完整提取成黑色区域，便于 OCR。

4. **前景/背景分离**
   - 视频监控中，背景建模后做差分，再自适应阈值，得到运动目标的二值掩码。
   - 这个掩码可以直接用于跟踪、计数、行为分析。

5. **医学图像分割**
   - CT/MRI 中，器官、病灶与周围组织灰度接近。
   - 自适应阈值能根据局部对比度把目标区域勾出来，供医生测量体积或做三维重建。

6. **作为更复杂算法的预处理**
   - 某些需要种子或标记的分割算法，通常先用自适应阈值得到粗略的前景区域，作为后续步骤的输入。
   - 需要初始化掩码的交互式分割方法，自适应阈值可以给一个粗略的前景/背景划分。
   - 连通组件分析、轮廓提取、霍夫变换等，几乎都依赖二值图作为输入。

#### 为什么“不连续”反而是优点？

因为**计算机不擅长处理连续值，擅长处理离散标签**。

- **连续灰度**：每个像素都是一个独立数值，计算机要判断“这个像素属于哪一类”，需要复杂的分类器。
- **二值标签**：每个像素只有“是/否”两种状态，计算机可以直接用图论、连通性、几何测量等成熟工具处理。

举个例子：
- 你想知道图里有几个硬币。
- 灰度图：你得先训练一个神经网络，或者做复杂的边缘检测+拟合。
- 自适应阈值后：直接 `cv2.connectedComponents()`，返回连通区域数量，就是硬币个数。简单、快速、可解释。

#### 那“丢失细节”怎么办？

你担心的“连续变不连续导致细节丢失”，在**分割任务**里其实是**有意的取舍**：

- 分割的目标不是保留所有细节，而是**突出感兴趣的区域，抑制不感兴趣的区域**。
- 自适应阈值通过局部比较，保留了**局部对比度**，丢掉的只是**绝对亮度信息**。
- 对于需要保留细节的任务（比如图像增强、风格迁移），确实不该用阈值化；但对于**测量、检测、识别**，阈值化是必要的第一步。

如果担心细节丢失太多，可以：
- 调整 `blockSize` 和 `C`，控制分割粒度。
- 用多阈值，保留多个灰度级。
- 在阈值化后做形态学开闭运算，平滑边界。
- 或者改用 K-means、Mean Shift 等聚类方法，得到更自然的分区。

#### 一句话总结

> **自适应阈值处理的目的，不是让图像“变好看”，而是把图像从“亮度连续场”转换成“区域离散标签”，从而让计算机能够计数、测量、检测和决策。**
> “不连续”不是副作用，而是**从感知到认知的关键一步**。

# 5. 移动平均阈值处理

## 5.1 移动平均原理

移动平均阈值是一种简单的自适应阈值方法。
它维护一个滑动窗口内的像素均值，作为当前像素的阈值。

OpenCV 没有直接提供该函数，但可以通过卷积或累积和实现。

### 5.1.1 这样做有什么好处？

移动平均阈值看起来和自适应均值阈值（`ADAPTIVE_THRESH_MEAN_C`）很像，但它的**工程意义**和**适用场景**有独特的优势。

**好处一：计算效率极高，适合实时场景**

- 滑动窗口可以**增量更新**：新窗口和 = 旧窗口和 - 移出的列 + 移入的列，每滑动一格只需 \( O(k) \) 次加减。
- OpenCV 的 `cv2.blur()` 内部使用**可分离滤波**（先横向均值，再纵向均值），复杂度是 \( O(k) \) 而不是 \( O(k^2) \)。
- **实际意义**：视频处理每帧都要分割，速度至关重要；嵌入式设备（树莓派、FPGA）算力有限；遥感、医学影像动辄几千万像素，\( O(N) \) 复杂度优势明显。

**好处二：实现简单，可解释性强**

- 没有“高斯核权重”这种额外参数，只有一个参数：**窗口大小**，物理意义明确。
- 便于手工实现和调试：在资源受限的环境（如单片机、DSP），可以自己写一个滑动窗口，逻辑简单、可控、容易移植。

**好处三：自然处理光照渐变**

- 扫描文档时，光源从一侧打过来，导致左边亮、右边暗。
- 固定阈值：右侧文字全部丢失。
- 移动平均：窗口滑到右侧时，局部均值也变低，阈值跟着变低，右侧文字仍能被正确提取。
- **为什么“滑动”比“分块”更好**：分块处理会在块边界产生**突变**，导致阈值不连续；移动平均是**逐像素滑动**，阈值是**连续变化**的，没有块边界伪影。

**好处四：与形态学、连通组件天然兼容**

- 移动平均得到的是**逐像素的局部阈值**，输出的二值图在局部是连续的，不会出现分块造成的“格子状”噪声。
- 这对后续操作很重要：连通组件分析不会因为块边界把同一个物体切成两半；轮廓提取边缘更平滑；形态学开闭运算能有效去除细小噪点。

**好处五：可以灵活调整“滞后量” C 来适应不同任务**

- **C = 0**：严格以局部均值为界，前景和背景各占一半。
- **C > 0**：阈值降低，更多像素被归为前景，适合“宁可多保留”的任务（如缺陷检测）。
- **C < 0**：阈值升高，更少像素被归为前景，适合“宁可少保留”的任务（如文本提取）。

**和自适应均值阈值的关系**

| 对比项 | 移动平均阈值 | 自适应均值阈值 |
|--------|-------------|---------------|
| 公式 | 几乎一样 | 几乎一样 |
| 计算方式 | 滑动窗口增量更新 | OpenCV 内部也是滑动 |
| 实现难度 | 可以手工实现 | 依赖 OpenCV |
| 参数 | 窗口大小、C | blockSize、C |
| 适用场景 | 嵌入式、实时、大图 | 通用 |

**本质上，移动平均阈值就是自适应均值阈值的一种“可手工实现的版本”**，两者在数学上等价，但在工程实现上有差异。

**局限与注意事项**

- **对噪声敏感**：均值对噪声没有抑制能力，椒盐噪声会拉偏阈值。此时高斯加权更好。
- **窗口大小难选**：太小对噪声敏感，太大接近全局阈值。需要根据目标尺寸调节。
- **边缘处理**：图像边界处窗口不完整，需要填充（padding）策略。
- **不适用于纹理复杂区域**：纹理区域局部均值波动大，容易产生碎块。

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img = cv2.imread('p&v/grayLena.bmp', cv2.IMREAD_GRAYSCALE)       # 以灰度模式读取图像
if img is None:                                                  # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

local_mean = cv2.blur(img, (15, 15))                             # 计算 15x15 邻域均值
th_moving_avg = np.where(img > local_mean - 5, 255, 0).astype(np.uint8)  # 像素大于局部均值减 5 则置 255

plt.figure(figsize=(10, 5))                                      # 创建画布
plt.subplot(1, 2, 1)                                             # 第 1 个子图
plt.imshow(img, cmap='gray')                                     # 显示原始灰度图
plt.title('Original Gray')                                       # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 2, 2)                                             # 第 2 个子图
plt.imshow(th_moving_avg, cmap='gray')                           # 显示移动平均阈值结果
plt.title('Moving Average Threshold')                            # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

# 6. HSV 颜色空间的阈值分割

## 6.1 HSV 颜色空间

HSV（Hue, Saturation, Value）比 RGB 更适合颜色分割。
OpenCV 中 H 范围是 0~179，S 和 V 范围是 0~255。

```python
dst = cv2.inRange(src, lowerb, upperb)
```

- `src`：输入图像。
- `lowerb`：下界。
- `upperb`：上界。
- 输出：在范围内的像素为 255，否则为 0。

**为什么用 `portraitGrass.png` 演示 HSV 分割？**

HSV 分割的核心优势是**把“颜色（Hue）”和“亮度（Value）”解耦**。这张图正好能体现这一点：

- **颜色鲜明**：女孩的红色上衣在 HSV 中 H 值集中在两端（0~10 和 170~180），S 值高，V 值适中；背景的绿色草地 H 值集中在 35~85 区间，和红色在 H 轴上距离很远，容易区分。
- **光照均匀**：户外自然光，红色上衣没有明显的过曝或阴影，H 值稳定，`inRange` 效果好。
- **多目标可提取**：可以提取红色上衣，也可以提取绿色草地，甚至提取白色裙子，适合演示“同一张图，不同 H 区间提取不同目标”。

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img_bgr = cv2.imread('p&v/portraitGrass.png')                    # 读取彩色图像
if img_bgr is None:                                              # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)               # 转换为 HSV 颜色空间

lower_red1 = np.array([0, 100, 100])                             # 红色下界 1（H 在 0~10）
upper_red1 = np.array([10, 255, 255])                            # 红色上界 1
lower_red2 = np.array([170, 100, 100])                           # 红色下界 2（H 在 170~180）
upper_red2 = np.array([180, 255, 255])                           # 红色上界 2

mask1 = cv2.inRange(img_hsv, lower_red1, upper_red1)             # 提取红色区域 1
mask2 = cv2.inRange(img_hsv, lower_red2, upper_red2)             # 提取红色区域 2
mask = cv2.bitwise_or(mask1, mask2)                              # 合并两个红色区域掩码

result = cv2.bitwise_and(img_bgr, img_bgr, mask=mask)            # 用掩码提取原图中的红色区域

plt.figure(figsize=(15, 5))                                      # 创建画布
plt.subplot(1, 3, 1)                                             # 第 1 个子图
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))             # 显示原图
plt.title('Original')                                            # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 3, 2)                                             # 第 2 个子图
plt.imshow(mask, cmap='gray')                                    # 显示掩码
plt.title('Red Mask')                                            # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 3, 3)                                             # 第 3 个子图
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))              # 显示提取结果
plt.title('Red Region')                                          # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

**预期效果：**

- **掩码（mask）**：只有红色上衣是白色，其他全部是黑色，非常干净。
- **提取结果（result）**：只有红色上衣保留，背景全黑。

这比使用 `cat.jpg` 时视觉结果直观得多，因为红色和绿色在 H 轴上距离很远，`inRange` 可以干净地分离。

### 6.1.1 顺便演示绿色提取

同一张图也可以很方便地提取草地：

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img_bgr = cv2.imread('p&v/portraitGrass.png')                    # 读取彩色图像
if img_bgr is None:                                              # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)               # 转换为 HSV 颜色空间

lower_green = np.array([35, 40, 40])                             # 绿色下界
upper_green = np.array([85, 255, 255])                           # 绿色上界

mask_green = cv2.inRange(img_hsv, lower_green, upper_green)      # 提取绿色区域
result_green = cv2.bitwise_and(img_bgr, img_bgr, mask=mask_green) # 用掩码提取绿色区域

plt.figure(figsize=(15, 5))                                      # 创建画布
plt.subplot(1, 3, 1)                                             # 第 1 个子图
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))             # 显示原图
plt.title('Original')                                            # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 3, 2)                                             # 第 2 个子图
plt.imshow(mask_green, cmap='gray')                              # 显示绿色掩码
plt.title('Green Mask')                                          # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 3, 3)                                             # 第 3 个子图
plt.imshow(cv2.cvtColor(result_green, cv2.COLOR_BGR2RGB))        # 显示提取结果
plt.title('Green Region')                                        # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

# 7. 区域生长与分离

## 7.1 区域生长原理

区域生长从种子点开始，将相似像素合并到同一区域。
OpenCV 没有直接函数，但可以通过连通组件或泛洪填充实现。

```python
cv2.floodFill(image, mask, seedPoint, newVal, loDiff, upDiff, flags)
```

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img = cv2.imread('p&v/grayLena.bmp', cv2.IMREAD_GRAYSCALE)       # 以灰度模式读取图像
if img is None:                                                  # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

img_copy = img.copy()                                            # 复制灰度图
h, w = img_copy.shape                                            # 获取图像高度和宽度
mask = np.zeros((h + 2, w + 2), np.uint8)                        # 创建掩码，比原图大 2 像素

cv2.floodFill(img_copy, mask, (100, 100), 255,                   # 从 (100,100) 开始泛洪填充
              loDiff=10, upDiff=10, flags=4)                     # 低差异 10，高差异 10，4 连通

plt.figure(figsize=(10, 5))                                      # 创建画布
plt.subplot(1, 2, 1)                                             # 第 1 个子图
plt.imshow(img, cmap='gray')                                     # 显示原始灰度图
plt.title('Original Gray')                                       # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 2, 2)                                             # 第 2 个子图
plt.imshow(img_copy, cmap='gray')                                # 显示区域生长结果
plt.title('Flood Fill Region Growing')                           # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

# 8. 区域分离与聚合

## 8.1 区域分离与聚合原理

区域分离与聚合（Split and Merge）是一种自顶向下的分割方法。
先将图像分成小块，如果不满足一致性则继续分裂；
然后将相邻的相似区域合并。

OpenCV 没有直接函数，通常用四叉树或连通组件实现。

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img = cv2.imread('p&v/grayLena.bmp', cv2.IMREAD_GRAYSCALE)       # 以灰度模式读取图像
if img is None:                                                  # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

ret, th_otsu = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)  # OTSU 二值化
ret, labels = cv2.connectedComponents(th_otsu)                   # 对二值图做连通组件标记
print('连通组件数量:', ret)                                      # 打印连通组件数量（含背景）

colors = np.random.randint(0, 255, size=(ret, 3), dtype=np.uint8)  # 为每个组件生成随机颜色
colors[0] = [0, 0, 0]                                            # 背景设为黑色
colored = colors[labels]                                         # 将标签映射为颜色

plt.figure(figsize=(10, 5))                                      # 创建画布
plt.subplot(1, 2, 1)                                             # 第 1 个子图
plt.imshow(th_otsu, cmap='gray')                                 # 显示二值图
plt.title('Binary Image')                                        # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 2, 2)                                             # 第 2 个子图
plt.imshow(cv2.cvtColor(colored, cv2.COLOR_BGR2RGB))             # 显示彩色连通组件
plt.title('Connected Components')                                # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

# 9. 超像素区域分割

## 9.1 超像素（Superpixels）简介

### 9.1.1 什么是超像素

超像素是把图像中**位置相邻、颜色/亮度/纹理相近**的像素合并成的小区域。它不是识别物体，而是先把图像“粗略分块”，用几百到几千个区域代替几十万到上百万个像素，为后续处理降维。

**核心价值**：减少计算量、保留边界信息、让后续算法在更有意义的单元上工作。

### 9.1.2 常见算法

| 算法 | 核心思路 | 特点 |
|------|----------|------|
| **SLIC** | 把 K-means 聚类搬到图像上，在 5 维空间 `[L,a,b,x,y]` 中聚类 | 速度快、边界贴合好、最常用 |
| **SEEDS** | 从规则网格出发，通过移动边界优化能量函数 | 速度快、边界规整 |
| **LSC** | 基于谱聚类，做线性近似加速 | 超像素均匀紧凑 |

其中 **SLICO** 是 SLIC 的零参数变体，会自动调整紧凑度，无需手动调参。

### 9.1.3 SLIC 是怎么做到的

1. **表示像素**：每个像素用 5 维向量 `[L, a, b, x, y]`，前三维表示颜色（CIELAB 空间），后两维是坐标。
2. **定义距离**：颜色差异和空间距离量纲不同，用归一化距离让二者公平参与：

   ```
   D = sqrt( (d_color/N_c)² + (d_space/N_s)² )
   ```

3. **限制搜索**：每个像素只在周围 `2N_s × 2N_s` 邻域内找最近聚类中心，避免和所有中心比较，大幅提速。
4. **迭代收敛**：反复更新聚类中心，通常 10 次左右稳定。

### 9.1.4 评价超像素的几个维度

- **边界贴合度**：边界是否贴合真实物体边缘
- **紧凑度**：形状是否规整、大小是否均匀
- **速度**：能否快速处理大图
- **可控性**：能否方便指定数量或大小

SLIC 在速度、贴合度和易用性之间平衡最好，因此成为事实标准，OpenCV 的 `ximgproc` 直接提供了它。

### 9.1.5 超像素与图像分割的关系

超像素是图像分割的一种，但属于**过分割（over-segmentation）**：它切得比真实物体更碎，只做粗分块，不知道什么是物体。

| 对比项 | 超像素分割 | 最终分割（后续章节） |
|--------|-----------|---------------------|
| 目的 | 把图过分割成许多小块 | 在种子间找精确边界 |
| 粒度 | 细碎、数量多（几百到几千） | 目标级、数量少 |
| 是否对应真实物体 | 不一定 | 是 |
| 是否需要人工引导 | 不需要，无监督 | 需要种子/标记 |
| 输出 | 超像素标签图 + 边界掩码 | 每个像素的归属标签 |

**超像素不能直接当分割结果**：它只是把颜色、位置相近的像素聚在一起，不会告诉你“这些块合起来是个人”。要得到真正的物体边界，还需要后续的分割步骤。

**一句话总结**：超像素是图像的“预分割”，把像素聚成颜色相近、边界合理的小块，SLIC 用改造过的 K-means 在颜色+空间联合空间中高效实现这一目标。

OpenCV 的 `ximgproc` 模块提供了这些函数，但需要额外安装 `opencv-contrib-python`。

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img_bgr = cv2.imread('p&v/portraitGrass.png')                    # 读取彩色图像
if img_bgr is None:                                              # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

try:                                                             # 尝试导入 ximgproc
    import cv2.ximgproc as ximgproc                              # 导入扩展模块
    slic = ximgproc.createSuperpixelSLIC(img_bgr,                 # 创建 SLIC 超像素对象
                                          algorithm=ximgproc.SLICO,  # 使用 SLICO 算法
                                          region_size=20)         # 区域大小 20
    slic.iterate(10)                                             # 迭代 10 次
    mask_slic = slic.getLabelContourMask()                       # 获取超像素边界掩码
    img_slic = cv2.bitwise_and(img_bgr, img_bgr, mask=cv2.bitwise_not(mask_slic))  # 在边界处变黑
    plt.figure(figsize=(10, 5))                                  # 创建画布
    plt.subplot(1, 2, 1)                                         # 第 1 个子图
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))         # 显示原图
    plt.title('Original')                                        # 设置标题
    plt.axis('off')                                              # 关闭坐标轴
    plt.subplot(1, 2, 2)                                         # 第 2 个子图
    plt.imshow(cv2.cvtColor(img_slic, cv2.COLOR_BGR2RGB))        # 显示超像素边界
    plt.title('SLIC Superpixels')                                # 设置标题
    plt.axis('off')                                              # 关闭坐标轴
    plt.tight_layout()                                           # 自动调整子图间距
    plt.show()                                                   # 显示图像
except ImportError:                                              # 如果未安装 ximgproc
    print('未安装 opencv-contrib-python，跳过超像素示例')        # 打印提示信息

# 10. 分水岭算法

## 10.1 分水岭原理

### 10.1.1 核心直觉：把图像当地形

分水岭算法把图像看作**地形**：

- 灰度值 = 海拔高度，亮处是山，暗处是谷
- 模拟往地形上注水，水先填满各个山谷形成湖
- 水位上涨，不同湖将接触时，在交界处修堤坝
- 这些堤坝就是**分界线**，每个湖就是一个区域

名字来自地形学：山脊线把不同水系分开，那道山脊就是分水岭。

### 10.1.2 直接使用的问题：过分割

直接把灰度图丢进算法，几乎每个局部小凹陷都会变成一个“湖”。真实图像噪声、纹理、微小起伏太多，结果被切成成千上万碎块，无法使用。这叫**过分割（over-segmentation）**。

解决思路不是改算法本身，而是**给它提供种子**：预先告诉算法哪些区域是确定的前景、哪些是确定的背景，让它只在种子之间注水。这就是下一节要讲的 marker（标记）。

### 10.1.3 标记（marker）的作用

OpenCV 的 `cv2.watershed()` 强制要求提供 `markers` 参数，本质是**你告诉算法：哪些区域是确定的前景、哪些是确定的背景、哪些还不确定**。

| 标记值 | 含义 |
|--------|------|
| 正数（1、2、3…） | 确定属于某个目标的种子区域 |
| 0 | 不确定区域，等算法分配 |
| -1 | 确定是边界，算法保留 |

算法只在种子之间“注水”，不会凭空产生小湖，从而避免过分割。

**种子由谁提供？** 由使用者（或程序）通过预处理手段主动构造，常见来源包括：阈值分割、距离变换、形态学膨胀、交互式鼠标点选等。具体怎么组合，下一节给出一个完整流程。

### 10.1.4 典型用法流程

1. **找确定背景**：阈值 + 膨胀，得到确定是背景的区域
2. **找确定前景**：距离变换 + 阈值，找到目标中心的种子
3. **找不确定区域**：前景和背景相减
4. **生成 markers**：背景标 1，前景标 2、3、4…，不确定区标 0
5. **调用 `cv2.watershed()`**：在种子间注水，输出归属
6. **可视化**：边界（-1）涂红，看到分割结果

### 10.1.5 找到分界线有什么用

分界线本身不是终点，而是把图像从**像素集合**变成**对象集合**，是后续任务的基础：

- **计数与测量**：数细胞、硬币、零件；算面积、周长、形状
- **分离粘连物体**：两个挨着的细胞、重叠零件，阈值法连成一片，分水岭能切开
- **为识别分类提供输入**：分割出肿瘤/地块/缺陷区域，再判断类型
- **抠图与背景替换**：定义前景轮廓，换背景、做合成
- **三维重建**：每层切片分割出轮廓，叠加重建三维结构
- **压缩与表示**：用“区域 + 属性”代替逐像素存储
- **引导后续算法**：配准、滤波、跟踪的约束单元

### 10.1.6 分水岭与图像分割的关系

图像分割是**任务/目标**（把图分成有意义的区域），分水岭是**实现这个目标的一种算法**。分割方法有很多：阈值、边缘、区域生长、聚类、图割、深度学习等，分水岭属于**基于形态学/拓扑**的那一类。

它的独特价值：

- 擅长**在已知种子之间找精确边界**
- 擅长**分开粘连物体**（阈值法做不到）
- 边界贴合梯度，往往比阈值法更准

但它不能独立完成分割——必须配合标记，而标记通常来自阈值、距离变换等其他手段。所以分水岭常是**分割流水线的最后一步**，负责“精修边界”。

### 10.1.7 一句话总结

分水岭把灰度当海拔、模拟注水找分界线；直接用它必然过分割，所以 OpenCV 要求先用 marker 标出确定的前景/背景/待定区，算法只在种子间找边界——这样才能得到干净、贴合轮廓、还能分开粘连物体的分割结果。而找到分界线的意义，就是**把图像从“像素集合”变成“对象集合”**，从而支撑计数、测量、分类、抠图、重建、压缩等几乎所有高层视觉任务。

### 10.1.8 分水岭与超像素的关系

现在分水岭已经讲完，可以和第 9 节的超像素做对比了。

| | 超像素 | 分水岭 |
|---|--------|--------|
| 性质 | 粗分割、预分割 | 在种子间找精确边界 |
| 特点 | 无监督降维 | 有引导的精确分割 |
| 输出 | 超像素标签图 + 边界掩码 | 每个像素的归属标签 |
| 共同点 | 都把像素组织成区域 | 同左 |

**两者互补**：

- 超像素擅长降维、去噪、提供规整的小单元。
- 分水岭擅长在种子间找精确边界、分开粘连物体。
- 超像素的缺点是边界不贴真实物体、不知道什么是目标。
- 分水岭的缺点是直接跑会过分割，需要种子引导。

典型流水线：

1. **超像素预分割**：用 SLIC 把图切成 N 个小块，计算量从百万像素降到几千块。
2. **在超像素上构建图**：每个超像素当一个节点，相邻超像素之间连边，边权用颜色差异衡量。
3. **生成种子**：通过阈值、距离变换或交互式标记，确定哪些超像素是前景、哪些是背景。
4. **分水岭精分割**：在这些种子之间找边界，把超像素合并成最终目标区域。

**一句话**：超像素提供“单元”，分水岭提供“边界”，两者配合，既快又准，共同完成从像素到对象的转换。

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img_bgr = cv2.imread('p&v/coin1.png')                            # 读取硬币图像（暗色背景 + 亮色硬币）
if img_bgr is None:                                              # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)                 # 转为灰度图

# ============ OTSU 全局阈值：亮硬币→白，暗背景→黑 ============
ret, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)  # OTSU 自动阈值：亮硬币判为前景

kernel = np.ones((3, 3), np.uint8)                               # 创建 3x3 结构元素
opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)   # 开运算：去掉背景中的小噪点
closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel, iterations=4) # 闭运算：填平硬币内部的黑洞（花纹、文字）
sure_bg = cv2.dilate(closing, kernel, iterations=3)              # 膨胀得到确定背景

dist_transform = cv2.distanceTransform(closing, cv2.DIST_L2, 5)  # 距离变换：硬币中心形成独立山峰
ret, sure_fg = cv2.threshold(dist_transform, 0.5 * dist_transform.max(), 255, 0)  # 确定前景：系数 0.5，切出硬币中心种子
sure_fg = np.uint8(sure_fg)                                      # 转为 uint8

# ============ 只保留较大的种子连通域，丢掉碎块 ============
num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(sure_fg, connectivity=8)  # 找所有种子连通域
areas = stats[1:, cv2.CC_STAT_AREA]                              # 各连通域面积（跳过背景标签 0）
if len(areas) > 0:                                               # 如果存在连通域
    max_area = areas.max()                                       # 最大连通域面积
    keep_labels = [i + 1 for i, a in enumerate(areas) if a >= 0.1 * max_area]  # 阈值 10%，保留硬币种子
    sure_fg_clean = np.zeros_like(sure_fg)                       # 创建全黑画布
    for lab in keep_labels:                                      # 逐个保留
        sure_fg_clean[labels == lab] = 255                       # 保留该连通域
    sure_fg = sure_fg_clean                                      # 替换原种子

unknown = cv2.subtract(sure_bg, sure_fg)                         # 未知区域 = 确定背景 - 确定前景

ret, markers = cv2.connectedComponents(sure_fg)                  # 连通组件标记：硬币一个种子标签
markers = markers + 1                                            # 背景设为 1（避开 0）
markers[unknown == 255] = 0                                      # 未知区域设为 0，等待分水岭分配

markers = cv2.watershed(img_bgr, markers)                        # 分水岭算法：在种子间注水，找边界

# ============ 可视化 ============
boundary = np.zeros_like(gray)                                   # 创建全黑画布
boundary[markers == -1] = 255                                    # 边界处画白（边界掩码）

boundary_thick = cv2.dilate(boundary, kernel, iterations=2)      # 膨胀加粗边界
img_with_boundary = img_bgr.copy()                               # 复制原图
img_with_boundary[boundary_thick > 0] = [0, 0, 255]              # 加粗边界涂红

mask_coins = (markers > 1).astype(np.uint8)                      # 标签 >1 的是硬币区域（背景是 1，边界是 -1）
img_coins = img_bgr * mask_coins[:, :, np.newaxis]               # 应用掩码，抠出硬币

# ============ 一行 4 张图 ============
plt.figure(figsize=(20, 5))                                      # 创建宽画布

plt.subplot(1, 4, 1)                                             # 第 1 张：种子
plt.imshow(sure_fg, cmap='gray')                                 # 显示确定前景（种子）
plt.title('Sure Foreground (Seeds)')                             # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 4, 2)                                             # 第 2 张：边界掩码
plt.imshow(boundary, cmap='gray')                                # 黑底白线显示边界
plt.title('Boundary Mask')                                       # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 4, 3)                                             # 第 3 张：原图 + 加粗边界
plt.imshow(cv2.cvtColor(img_with_boundary, cv2.COLOR_BGR2RGB))   # 显示原图（边界已加粗涂红）
plt.title('Original + Thick Boundaries')                         # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 4, 4)                                             # 第 4 张：抠出的硬币
plt.imshow(cv2.cvtColor(img_coins, cv2.COLOR_BGR2RGB))           # 显示抠出的硬币（背景黑）
plt.title('Coins Extracted')                                     # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

分水岭算法结果解读

一、四张图分别是什么

第 1 张：Sure Foreground (Seeds) —— 确定前景（种子）

- 画面：黑底上一个规整的白色椭圆，位于硬币中心。
- 含义：这是分水岭的"起点"。程序通过距离变换找到硬币内部离边缘最远的区域，用一个阈值切出一小块，作为该目标的种子。
- 作用：分水岭需要知道"图里有几个目标、各自在哪"，种子就是告诉它"这里有一个目标，从这里开始注水"。

第 2 张：Boundary Mask —— 边界掩码

- 画面：黑底上一圈完整、光滑、闭合的白色圆线。
- 含义：这就是分水岭最终找到的分界线，即 `markers == -1` 的那些像素。
- 作用：单独显示边界本身，不受原图纹理干扰，能最清楚地看到分割轮廓。

第 3 张：Original + Thick Boundaries —— 原图 + 加粗边界

- 画面：原硬币图上，硬币边缘叠了一圈粗红圈。
- 含义：把边界加粗后涂红，叠加回原图，方便肉眼确认边界位置是否准确。
- 作用：验证边界是否贴合硬币真实外缘。

第 4 张：Coins Extracted —— 抠出的硬币

- 画面：黑背景上一枚完整的硬币。
- 含义：用分水岭的标签当掩码，把硬币区域保留、背景涂黑。
- 作用：分割的最终产物——从原图中把目标完整抠出来。

二、评价：这是理想结果

三条硬指标全部满足：

| 指标 | 状态 |
|------|------|
| 每个目标都有独立、规整的种子 | 满足，硬币中心一个白椭圆 |
| 边界闭合、贴合真实轮廓 | 满足，红圈完整贴住外缘 |
| 抠图完整、背景干净 | 满足，硬币完整、背景全黑 |

具体来说：

- 种子形状规整，位于硬币正中，没有缺口或碎块。
- 边界线光滑闭合，没有锯齿、毛刺、多余分支。
- 红圈紧贴硬币外缘，既不咬进硬币内部，也不跑到背景里。
- 抠图干净利落，硬币本体一点不缺，背景全黑。

三、为什么这张图能成功

关键在于输入图的灰度对比足够干净：

- 背景暗、硬币亮，两者界限分明。
- 硬币内部纹理虽复杂，但闭运算能填平，不影响种子。
- 硬币外缘与背景对比强，OTSU 一刀切下去就能把硬币完整分出来。

对比之前失败的几张图：

| 图 | 问题 | 结果 |
|----|------|------|
| portraitGrass | 前景背景混杂，二值化方向反了 | 失败 |
| coin.jpg | 硬币边缘有阴影，种子歪 | 左硬币失败 |
| coin2 / coin3 | 硬币和背景灰度太接近 | 只分出一部分 |
| coin1.jpg | 暗背景 + 亮硬币，对比明确 | 完美 |

四、结论

这次是理想的分割结果。它说明：

- 分水岭算法本身没有问题。
- 成败的关键在于输入图的前景背景对比是否干净。
- 对比明确时，种子规整、边界圆滑、抠图完整，一次到位。

这张 coin1.jpg 可以作为分水岭演示的正式样例。

# 11. GrabCut算法

## 11.1 GrabCut 原理

GrabCut 是一种交互式前景提取算法。
用户指定一个矩形框，算法迭代地分割前景和背景。

```python
cv2.grabCut(img, mask, rect, bgdModel, fgdModel, iterCount, mode)
```

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img_bgr = cv2.imread('p&v/portraitGrass.png')                    # 读取彩色图像
if img_bgr is None:                                              # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

mask = np.zeros(img_bgr.shape[:2], np.uint8)                     # 创建全零掩码
bgdModel = np.zeros((1, 65), np.float64)                         # 背景模型
fgdModel = np.zeros((1, 65), np.float64)                         # 前景模型
rect = (50, 50, img_bgr.shape[1] - 100, img_bgr.shape[0] - 100)  # 定义矩形框（四周各留 50 像素）

cv2.grabCut(img_bgr, mask, rect, bgdModel, fgdModel, 5, cv2.GC_INIT_WITH_RECT)  # GrabCut 分割
mask2 = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')  # 提取前景掩码
img_grabcut = img_bgr * mask2[:, :, np.newaxis]                  # 应用掩码，抠出前景

# ============ 在原图上画出用户指定的矩形框 ============
img_with_rect = img_bgr.copy()                                   # 复制原图，用于画框
x, y, w, h = rect                                                # 拆出矩形左上角坐标和宽高
cv2.rectangle(img_with_rect, (x, y), (x + w, y + h), (0, 0, 255), 3)  # 画红色矩形框，线宽 3 像素

# ============ 一行 3 张图 ============
plt.figure(figsize=(18, 6))                                      # 创建宽画布

plt.subplot(1, 3, 1)                                             # 第 1 张：原图
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))             # 显示原图
plt.title('Original')                                            # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 3, 2)                                             # 第 2 张：原图 + 矩形框
plt.imshow(cv2.cvtColor(img_with_rect, cv2.COLOR_BGR2RGB))       # 显示带矩形框的原图
plt.title('Original + Rect')                                     # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.subplot(1, 3, 3)                                             # 第 3 张：抠图结果
plt.imshow(cv2.cvtColor(img_grabcut, cv2.COLOR_BGR2RGB))         # 显示 GrabCut 抠图结果
plt.title('GrabCut')                                             # 设置标题
plt.axis('off')                                                  # 关闭坐标轴

plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

矩形框是 **GrabCut 的"初始提示"**——它告诉算法："**框内大概是前景，框外大概是背景，你从这开始迭代。**"

**具体作用**

GrabCut 是**交互式**分割算法，它不是全自动的，需要用户先给一个"粗略指引"。矩形框就是这个指引，作用分三步：

**第一步：框外直接标为"确定背景"**

矩形框外的所有像素（比如右上角的树木、左侧和底部的草地），GrabCut **直接判定为背景**，不再考虑。

**第二步：框内先标为"可能是前景"**

矩形框内的所有像素（人物、部分草地、部分裙子），GrabCut 先**暂时**都当成"可能是前景"，作为初始猜测。

**第三步：迭代优化**

GrabCut 在框内做**迭代**（代码里是 5 次）：

- 用颜色统计建前景/背景模型；
- 把框内颜色像前景的留下、像背景的踢出去；
- 逐步缩小前景范围，直到稳定。

比如框内底部的草地，虽然一开始被当成"可能是前景"，但迭代几次后，算法发现它的颜色和框外草地（确定背景）一样，就把它踢出去了。最终只有人物留下。

**为什么需要这个框**

GrabCut 不能"无中生有"地知道哪个是前景。你给它一个**大致范围**，它就能在这个范围内细化边界。

- **框太紧**：比如只框住脸，算法就只抠脸，肩膀、裙子全丢。
- **框太松**：比如框住整张图，框外没有"确定背景"做对比，算法容易出错。
- **框得合适**：框住人物、四周留一圈背景，效果最好。

你这张图里 `rect = (50, 50, 宽-100, 高-100)` 就是四周各留 50 像素边距，正好把人物框住、又留了一圈背景做参照。

**框和最终结果的关系**

观察你这三张图：

- **第 2 张**：红框框住人物 + 一部分草地。
- **第 3 张**：GrabCut 只抠出**人物**，草地全被踢掉了。

这说明迭代成功了：**框内的草地虽然在初始范围内，但因为颜色和框外草地一致，被算法判为背景剔除了。**

**一句话**

矩形框的作用是**给 GrabCut 一个"起点范围"**：

- 框外 = 确定背景，直接排除；
- 框内 = 可能是前景，迭代细化。

算法在这个框架下，用颜色统计和迭代，最终把真正的前景（人物）从框内的背景（草地）中分离出来。**框只是引导，不是结果。**

右上角（人物头发右上方）确实缺了一块，被涂成了黑色，说明 GrabCut 把那块头发误判成了背景。


原因是：矩形框右边界切到了头发，有一部分超出了矩形框的右边界。

超出框的部分，按 GrabCut 的规则，直接被判定为"确定背景"，不可能被当成前景。

所以那块头发从一开头就被排除，后面的迭代再准也救不回来。

# 12. 均值漂移算法

## 12.1 均值漂移原理

均值漂移是一种聚类算法，用于图像平滑和分割。
`cv2.pyrMeanShiftFiltering()` 在金字塔上进行均值漂移滤波。

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

img_bgr = cv2.imread('p&v/portraitGrass.png')                    # 读取彩色图像
if img_bgr is None:                                              # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误

img_shift = cv2.pyrMeanShiftFiltering(img_bgr, sp=10, sr=30)     # 均值漂移滤波，空间半径 10，颜色半径 30

plt.figure(figsize=(10, 5))                                      # 创建画布
plt.subplot(1, 2, 1)                                             # 第 1 个子图
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))             # 显示原图
plt.title('Original')                                            # 设置标题
plt.axis('off')                                                  # 关闭坐标轴
plt.subplot(1, 2, 2)                                             # 第 2 个子图
plt.imshow(cv2.cvtColor(img_shift, cv2.COLOR_BGR2RGB))           # 显示均值漂移结果
plt.title('Mean Shift')                                          # 设置标题
plt.axis('off')                                                  # 关闭坐标轴
plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

**均值漂移分割结果解读**

**一、两张图分别是什么**

- 左图 Original：原始图像，草地、头发、裙子、树叶都保留着细密的纹理和噪点。
- 右图 Mean Shift：经过 `cv2.pyrMeanShiftFiltering()` 处理后的图像，颜色相近的区域被抹平，细纹理被去掉，整体呈现"色块化"效果。

**二、为什么看起来"没分割"**

均值漂移的输出**不是黑白分割图，也不是画边界线的图**，而是**保持原图颜色的"色块化"彩色图**。所以粗看像"加了油画滤镜"，而不是"分成几块"。

**三、右图具体发生了什么变化**

- 草地：左图能看到一根根草、小凸起、细纹理；右图草地变成大片均匀的绿色，细纹理没了。
- 裙子：左图条纹、波浪纹边缘锐利；右图条纹被平滑掉一部分，颜色块更"糊"。
- 头发：左图头发丝根根分明；右图头发变成几块相近的棕黄色。
- 红上衣：左图红色有深浅变化；右图红得更均匀。
- 背景树：左图树叶细节多；右图树叶变成一块块深绿浅绿。

**四、均值漂移的"分割"和别的分割有什么区别**

| 方法 | 输出形式 | 视觉效果 |
|------|----------|----------|
| 阈值 | 黑白二值图 | 硬边界，目标变白块 |
| 分水岭 | 原图 + 边界线 | 目标边缘画线 |
| 均值漂移 | 色块化彩色图 | 区域内部变平，边缘保留 |
| GrabCut | 抠图（背景黑） | 目标留下、背景消失 |

**五、均值漂移在分割里的定位**

均值漂移属于**过分割**的一种。它把图分成很多颜色相近的小区域，但这些区域**不一定对应真实物体**，只是颜色相近而已。它更像是分割的前置步骤，帮你把图简化成色块，方便后续再处理。

**六、一句话总结**

右图已经"分割"了——它把图聚类成了许多颜色相近的色块，只是用颜色表示区域，而不是画黑线。草地、头发、裙子内部都被抹平了，这就是均值漂移的效果。它看起来像"加了油画滤镜"，而不是"分成几块"，因为它的分割方式是"色块化"，不是"画边界"。

# 13. 运动图像分割

**目标**：从视频或图像序列中检测并分割出运动目标。

**说明**："利用时间维度的分割"——运动目标靠帧与帧之间的差异来定位，而不是靠单帧的颜色或灰度。

**本章方法**：
- 帧间差分法：比较相邻帧差异
- 背景差分法：建立背景模型，当前帧减背景
- 密集光流法：计算每个像素的运动向量

**适用场景**：视频监控、行为分析、交通流量统计。

## 13.1 帧间差分法

帧间差分法通过比较相邻帧的差异来检测运动。

```python
diff = cv2.absdiff(frame1, frame2)
```

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作

frame1 = cv2.imread('p&v/ss/s1.png', cv2.IMREAD_GRAYSCALE)          # 第一帧
frame2 = cv2.imread('p&v/ss/s2.png', cv2.IMREAD_GRAYSCALE)          # 第二帧
if frame1 is None or frame2 is None:                             # 如果读取失败
    raise FileNotFoundError('图像读取失败，请检查路径')          # 抛出异常，提示路径错误



diff = cv2.absdiff(frame1, frame2)                               # 计算帧间差分：运动区域差异大，静止区域差异接近 0
ret, th_diff = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)   # 二值化差分结果，突出运动区域

plt.figure(figsize=(15, 5))                                      # 创建画布
plt.subplot(1, 3, 1)                                             # 第 1 个子图
plt.imshow(frame1, cmap='gray')                                  # 显示第一帧
plt.title('Frame 1')                                             # 设置标题
plt.axis('off')                                                  # 关闭坐标轴
plt.subplot(1, 3, 2)                                             # 第 2 个子图
plt.imshow(frame2, cmap='gray')                                  # 显示第二帧
plt.title('Frame 2')                                             # 设置标题
plt.axis('off')                                                  # 关闭坐标轴
plt.subplot(1, 3, 3)                                             # 第 3 个子图
plt.imshow(th_diff, cmap='gray')                                 # 显示差分二值图（白色 = 运动区域）
plt.title('Frame Difference')                                    # 设置标题
plt.axis('off')                                                  # 关闭坐标轴
plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

## 13.2 背景差分法

背景差分法通过建立背景模型，将当前帧与背景相减来检测运动。
OpenCV 提供了 `cv2.createBackgroundSubtractorMOG2()` 等。


**背景差分法原理**

**一、它做什么**

背景差分法通过建立背景模型，将当前帧与背景相减来检测运动。
OpenCV 提供了 `cv2.createBackgroundSubtractorMOG2()` 等接口。

**二、背景模型是什么**

背景模型不是一张固定的背景图，而是对**每个像素**估计"如果没有任何运动目标，这个像素应该是多少"。

它记录的是每个像素的**历史统计规律**：
- 如果某个像素在历史上长期是某个值，那这个值就是它的背景。
- 当某帧这个像素突然偏离历史规律，就判为前景（运动目标）。

**三、背景像素会波动**

即使相机固定、没有运动目标，同一个背景像素在不同帧里也会变化：

- 光照变化：云飘过、灯闪烁，背景整体亮度小幅变动。
- 树叶被风吹：树叶区域像素在亮绿/暗绿之间跳。
- 水面反光：波光区域像素忽明忽暗。
- 显示器刷新：屏幕区域像素周期性变化。
- 传感器噪声：每个像素本身就有随机抖动。

**所以"背景像素值"不是恒定常数，而是一个有波动的统计分布。**

**四、背景怎么估计：混合高斯模型（MOG2）**

MOG2 对每个像素，用**多个高斯分布**描述它历史上的取值：

- 落在权重大的分布里 → 认为是背景。
- 落在权重小的分布里，或偏离所有分布 → 认为是前景。

用"多个"高斯而不是一个，是因为真实场景里像素值常常是**多模态**的：树叶在两种颜色间跳、水面在明暗间闪、显示器周期性刷新，单高斯描述不了，必须用多个高斯分别描述。

MOG2 的"2"是 Zivkovic 对原版 MOG 的改进：

- 自动为每个像素决定"该用几个高斯"，不用手写 K。
- 对光照变化更鲁棒。
- 参数更少，更好用。

**五、为什么比"一张固定背景图"鲁棒**

如果只拿一张固定背景图去减：

- 光照稍微一变，全图产生差异 → 整张图误判成前景。
- 树叶一晃，树叶区域误判。
- 水面一闪，水面区域误判。

而 MOG2 用多个高斯分布记录每个像素的波动规律：

- 光照在正常范围内波动 → 落在背景分布里 → 正确判为背景。
- 树叶在两种颜色间跳 → 两个高斯分布分别描述 → 都算背景。
- 只有当像素**偏离历史规律**（比如静止墙面突然出现深色人形），才判为前景。

**这就是"鲁棒"的含义：能容忍背景的正常波动，只对真正异常的变化报警。**

**六、和帧间差分法的区别**

| | 帧间差分 | 背景差分（MOG2） |
|---|----------|------------------|
| 参照物 | 前一帧 | 多帧统计出的背景分布 |
| 背景波动 | 无法区分"波动"和"运动" | 把波动学进模型，不误报 |
| 鲁棒性 | 低 | 高 |
| 目标内部空洞 | 常见 | 能消除 |
| 缓慢移动目标 | 不敏感 | 更敏感 |

帧间差分只比较相邻两帧，任何波动都会被当成差异。背景差分用长期统计，能分清正常波动和真实运动。

**七、适用条件**

背景差分法**要求相机静止**。

- 相机静止、背景稳定 → 适合背景差分。
- 相机移动、背景一直在变 → 背景模型失效，应改用光流法或运动补偿。

**八、一句话总结**

背景像素在帧间是会波动的；MOG2 用多个高斯分布把这些正常波动学进模型，所以比"一张固定背景图"鲁棒得多。它只在像素"偏离历史规律"时才判为前景——这正是背景差分法比帧间差分法强的地方。

In [ ]:
import cv2                        # 导入 OpenCV，模块名为 cv2
import matplotlib.pyplot as plt   # 导入 Matplotlib 的 pyplot，用于绘图展示
import numpy as np                # 导入 NumPy，用于数值计算与数组操作
import os                         # 导入 os，用于遍历目录

frame_dir = 'p&v/SS'                                             # 多帧所在目录
frame_files = sorted(os.listdir(frame_dir),                      # 列出目录下所有文件并按名称排序
                     key=lambda x: int(x[1:-4]))                 # 按 S 后面的数字排序：S1、S2、…、S10
if len(frame_files) == 0:                                        # 如果目录为空
    raise FileNotFoundError('SS 目录下没有帧文件')               # 抛出异常，提示无帧

bg_subtractor = cv2.createBackgroundSubtractorMOG2()             # 创建 MOG2 背景减除器

last_frame = None                                                # 最后一帧原图
last_fg_mask = None                                              # 最后一帧前景掩码

for filename in frame_files:                                     # 逐帧遍历
    path = os.path.join(frame_dir, filename)                     # 拼接完整路径
    frame = cv2.imread(path, cv2.IMREAD_GRAYSCALE)               # 以灰度模式读取该帧
    if frame is None:                                            # 如果读取失败
        continue                                                 # 跳过该帧
    fg_mask = bg_subtractor.apply(frame)                         # 把当前帧喂给 MOG2，得到前景掩码
    last_frame = frame                                           # 记录当前帧
    last_fg_mask = fg_mask                                       # 记录当前前景掩码

print(f'共处理 {len(frame_files)} 帧')                           # 打印处理帧数

plt.figure(figsize=(10, 5))                                      # 创建画布
plt.subplot(1, 2, 1)                                             # 第 1 个子图
plt.imshow(last_frame, cmap='gray')                              # 显示最后一帧原图
plt.title('Last Frame')                                          # 设置标题
plt.axis('off')                                                  # 关闭坐标轴
plt.subplot(1, 2, 2)                                             # 第 2 个子图
plt.imshow(last_fg_mask, cmap='gray')                            # 显示最后一帧前景掩码
plt.title('Background Subtraction')                              # 设置标题
plt.axis('off')                                                  # 关闭坐标轴
plt.tight_layout()                                               # 自动调整子图间距
plt.show()                                                       # 显示图像

**背景差分法结果解读：50 帧是怎么用的，最后检测的是什么**

**一、代码在做什么**

代码从 `p&v/SS/` 目录里逐帧读取 S1、S2、…、S50，每帧都喂给 `cv2.createBackgroundSubtractorMOG2()`，最后显示**第 50 帧**的原图和前景掩码。

**二、50 帧不是"两两配对做差分"**

很多人会误以为 50 帧是"第 1 帧减第 2 帧、第 2 帧减第 3 帧……"——那是**帧间差分法**，不是背景差分法。

背景差分法里，50 帧的作用是**喂给 MOG2 学背景**：

- 每帧进来，MOG2 对每个像素更新一次统计分布。
- 不动的像素（楼、路、草地）值长期稳定，被学成"背景"。
- 走动的像素（行人位置）值一直变，学不成稳定背景。
- 50 帧结束后，MOG2 内部形成一张"背景模型"——对每个像素知道"正常情况下它应该是什么值"。

**三、最后检测的是"哪一帧和背景的差异"**

每一帧喂给 `apply()` 时，都会返回"这一帧相对当前背景模型的差异"：

- 第 1 帧：背景还没学好，`fg_mask` 很乱。
- 第 25 帧：背景学了一半，`fg_mask` 开始稳定。
- **第 50 帧：背景学好了，`fg_mask` 就是"第 50 帧减去学出来的背景模型"。**

代码最后显示的，就是第 50 帧的 `fg_mask`。

**用一句话概括：**

> **第 50 帧 − MOG2 学出来的背景模型 = 前景掩码**

**四、右图里白色的都是什么**

右图 `Background Subtraction` 里的白色区域，对应左图里**正在移动的人**：

| 右图位置 | 左图对应 |
|----------|----------|
| 中上部一排白色人形 | 几个人站在/走过路口 |
| 中右部两个白色人形 | 两个正在走的人 |
| 右侧一个白色人形 | 最右边单独走的人 |
| 零散小白点 | 噪声、树叶/旗帜晃动等误检 |

**白色 = 相对于背景发生了变化的像素**，也就是"动了的东西"。

**五、为什么站着的人没被检测出来**

左图中部偏左有两个人**基本站着不动**，右图里没有他们。原因是：

- 他们长期不动，MOG2 把他们学成了"背景的一部分"。
- 背景差分法检测的是"运动"，不是"人"。

**这说明：不动的人、静止的楼和车，都会被学成背景，显示为黑色。**

**六、和帧间差分法的区别**

| | 帧间差分 | 背景差分（MOG2） |
|---|----------|------------------|
| 比较对象 | 相邻两帧 | 当前帧 vs 学出来的背景模型 |
| 参照物 | 动态的（前一帧） | 稳定的（统计背景） |
| 目标内部空洞 | 常见 | 能消除 |
| 缓慢移动目标 | 不敏感 | 更敏感 |
| 站着不动的人 | 可能被检测（如果两帧间有微动） | 学成背景，不检测 |

**七、那些零散小白点是什么**

右图里还有些零星小白点（比如左上角、中上部虚线状的点）：

- 噪声：传感器随机噪声、压缩伪影。
- 背景微动：树叶被风吹、旗帜飘动、水面反光。
- 误检：MOG2 还没完全学稳，或阈值敏感导致。



**八、一句话总结**

50 帧是用来"喂" MOG2 学背景的，最后检测的是"第 50 帧减去学出来的背景模型"。不是 50 帧两两差分，而是"最后一帧 vs 统计出来的背景"。右图里每个白色人形，对应左图里一个运动中的行人；站着不动的人、静止的楼和车，都被学成背景、显示为黑色。**背景差分法检测的是"运动区域"，不是"人"本身。**

## 13.3 密集光流法

密集光流法计算每个像素的运动向量。
OpenCV 提供了 `cv2.calcOpticalFlowFarneback()`。


**一、光流是什么**

光流是图像中每个像素的"运动速度"。对每个像素，估计它从第 1 帧到第 2 帧移动了多少，用向量表示：

$$
\vec{v}(x, y) = \bigl(u(x, y),\ v(x, y)\bigr)
$$

其中：

- $u(x, y)$：水平方向位移
- $v(x, y)$：垂直方向位移

一整张图所有像素的向量，就是**光流场**。

**二、光流的三个假设**

1. **亮度恒定**：同一个物体点，在两帧里亮度不变，只是位置移动了。
2. **小运动**：两帧之间位移小（几个像素内）。
3. **空间一致**：相邻像素运动方向相近。

**三、光流方程与孔径问题**

设 $I(x, y, t)$ 为图像在位置 $(x, y)$、时刻 $t$ 的灰度。由"亮度恒定"假设，物体点移动后灰度不变：

$$
I(x + u,\ y + v,\ t + 1) = I(x,\ y,\ t)
$$

对左边做一阶泰勒展开：

$$
I(x,\ y,\ t) + I_x u + I_y v + I_t = I(x,\ y,\ t)
$$

消去 $I(x, y, t)$，得到**光流方程**：

$$
I_x\, u + I_y\, v + I_t = 0
$$

其中：

- $I_x = \dfrac{\partial I}{\partial x}$：水平方向梯度
- $I_y = \dfrac{\partial I}{\partial y}$：垂直方向梯度
- $I_t = \dfrac{\partial I}{\partial t}$：时间方向梯度（两帧灰度差）
- $u,\ v$：待求的水平、垂直位移

**问题**：一个方程、两个未知数 $u, v$，**解不出来**——这叫**孔径问题**（aperture problem）。必须额外加约束才能求解。

**四、Farneback 怎么解**

Farneback 用**多项式展开**近似图像。对每个像素邻域，用一个二次多项式拟合：

$$
f(\mathbf{x}) \approx \mathbf{x}^\top \mathbf{A}\,\mathbf{x} + \mathbf{b}^\top \mathbf{x} + c
$$

其中 $\mathbf{x} = (x, y)^\top$，$\mathbf{A}$ 是对称矩阵，$\mathbf{b}$ 是向量，$c$ 是常数。

两帧的多项式系数一对比，就能解出位移 $\mathbf{d} = (u, v)^\top$：

$$
\mathbf{A}_2 = \mathbf{A}_1,\qquad
\mathbf{b}_2 = \mathbf{b}_1 - 2\,\mathbf{A}_1\,\mathbf{d}
$$

由第二式反解出 $\mathbf{d}$：

$$
\mathbf{d} = -\tfrac{1}{2}\,\mathbf{A}_1^{-1}\,(\mathbf{b}_2 - \mathbf{b}_1)
$$

为了处理大位移，Farneback 用**图像金字塔**从粗到细逐层计算：先在缩小的图上估大位移，再逐层细化。

**五、密集 vs 稀疏**

| 类型 | 代表算法 | 计算对象 | 优点 | 缺点 |
|------|----------|----------|------|------|
| 稀疏光流 | Lucas-Kanade | 少数特征点（角点） | 快 | 覆盖不全 |
| 密集光流 | Farneback | 每个像素 | 得到完整运动场 | 慢 |

**六、输出**

Farneback 返回一个和图像同尺寸的 $(H, W, 2)$ 数组：

- $\text{flow}[y, x, 0] = u$：该像素水平位移
- $\text{flow}[y, x, 1] = v$：该像素垂直位移

用 `cv2.cartToPolar` 可以转成极坐标：

$$
\text{mag} = \sqrt{u^2 + v^2},\qquad \text{ang} = \operatorname{atan2}(v,\ u)
$$

- **mag**：运动幅度（越亮表示运动越剧烈）
- **ang**：运动方向

**一句话**
光流 = 每个像素的运动向量 (dx, dy)；由"亮度恒定"推出光流方程，但一个方程解不出两个未知数（孔径问题）；Farneback 用多项式展开 + 图像金字塔求解，得到整张图的运动场。

**三种运动检测方法对比：帧间差分法、背景差分法、密集光流法**

**一、它们各自在做什么**

**帧间差分法**

把相邻两帧逐像素相减，差异大的地方判为"有运动"。

$$D(x, y) = |I_{t+1}(x, y) - I_t(x, y)|$$

- 输入：**2 帧**
- 输出：**二值差异图**（哪里变了）
- 特点：最简单、最快

**背景差分法**

先学出一个"背景模型"，再用当前帧减背景。

$$D(x, y) = |I_t(x, y) - B(x, y)|$$

- 输入：**多帧**（几十帧）
- 输出：**二值前景掩码**（哪里相对背景在动）
- 特点：需要相机静止

**密集光流法**

对每个像素，算它从上一帧到下一帧的**运动向量**。

$$\vec{v}(x, y) = \bigl(u(x, y),\ v(x, y)\bigr)$$

- 输入：**2 帧**
- 输出：**运动向量场**（每个像素的方向 + 距离）
- 特点：信息最丰富，计算量最大

**二、和帧间差分法的核心区别**

帧间差分法和密集光流法**都用 2 帧**，但本质完全不同：

| 对比项     | 帧间差分法           | 密集光流法                   |
| ---------- | -------------------- | ---------------------------- |
| 核心操作   | 逐像素**相减**       | 逐像素**求运动向量**         |
| 建方程吗   | 不建                 | 建（光流方程）               |
| 输出       | 差异值（标量）       | 运动向量（两个数 dx, dy）    |
| 知道方向吗 | **不知道**           | **知道**                     |
| 知道距离吗 | **不知道**           | **知道**                     |
| 目标形状   | **空心轮廓**         | **实心区域**                 |
| 背景噪声   | 多（微小变化也误检） | 少（能区分真运动和亮度变化） |
| 运动强度   | 只有"变/没变"        | 能给出**连续数值**           |
| 计算量     | 小                   | **大**                       |

**三、帧间差分法为什么"不够好"**

三个天生缺陷：

**缺陷一：空心**

人移动时，人的边缘在两帧间值变了（白），人的内部两帧都是人（黑）。所以检测出来是**空心轮廓**，不是实心人形。

**缺陷二：噪声多**

树叶晃动、光照波动、压缩噪声，两帧间都有微小差异，都会被误检为"运动"。

**缺陷三：分不清"运动"和"亮度变化"**

灯亮了一下，值也变了，也被判为"有运动"。

**四、密集光流法的优势**

**优势一：能给出方向和距离**

帧间差分只知道"这里变了"；光流知道"这里向右移了 5 像素"。这是**跟踪、测速、行为分析**的基础。

**优势二：目标完整不空心**

光流对每个像素都算运动向量，人内部也在动，所以整个人的区域都有值，是**实心区域**。

**优势三：能区分"真运动"和"亮度变化"**

光流通过"对应匹配"工作，亮度变化不会产生运动向量，所以噪声少。

**优势四：能给出运动强度**

光流幅度是连续值，可以显示"运动多剧烈"；帧间差分只能给"变/没变"。

**五、密集光流法的代价**

**代价一：计算量大**

对每个像素都要解方程、做金字塔迭代，比帧间差分慢几十倍。

**代价二：假设苛刻**

要求"亮度恒定、小运动、空间一致"。位移太大、光照剧变、纯色区域，光流都算不准。

**代价三：输出不是"目标掩码"**

光流给的是"运动强度图"，要得到"目标在哪"还得再阈值化。

**六、三种方法适用场景**

| 方法       | 适合场景                           | 不适合场景                     |
| ---------- | ---------------------------------- | ------------------------------ |
| 帧间差分法 | 相机固定、算力有限、只要"哪里动了" | 需要方向、需要完整目标、噪声多 |
| 背景差分法 | 相机固定、背景稳定、要完整前景     | 相机移动、背景一直在变         |
| 密集光流法 | 需要方向、需要测速、需要完整运动场 | 算力有限、位移过大、纯色区域   |

**七、一句话总结**

- **帧间差分法**：只告诉你"**哪里变了**"，最快，但空心、噪声多、不知道方向。
- **背景差分法**：告诉你"**哪里相对背景在动**"，需要多帧学背景，结果干净，但要求相机静止。
- **密集光流法**：告诉你"**每个像素往哪动、动多少**"，信息最丰富、结果最完整，但计算量大、假设苛刻。

**帧间差分是"相减"，光流是"匹配对应点"——前者只知道变化，后者知道运动的方向和大小。**

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

frame1 = cv2.imread('p&v/SS/S1.png', cv2.IMREAD_GRAYSCALE)       # 第一帧
frame2 = cv2.imread('p&v/SS/S2.png', cv2.IMREAD_GRAYSCALE)       # 第二帧
if frame1 is None or frame2 is None:
    raise FileNotFoundError('图像读取失败，请检查路径')

flow = cv2.calcOpticalFlowFarneback(frame1, frame2, None,        # Farneback 密集光流
                                    0.5, 3, 15, 3, 5, 1.2, 0)
mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])           # 幅度 + 角度

fig = plt.figure(figsize=(18, 5))                                # 创建画布

# 三张图的位置和大小完全一致：宽 0.28、高 0.85
ax1 = fig.add_axes([0.02, 0.07, 0.28, 0.85])                     # 第 1 张
ax2 = fig.add_axes([0.34, 0.07, 0.28, 0.85])                     # 第 2 张
ax3 = fig.add_axes([0.66, 0.07, 0.28, 0.85])                     # 第 3 张
cax = fig.add_axes([0.955, 0.07, 0.015, 0.85])                   # colorbar 单独一条

ax1.imshow(frame1, cmap='gray')                                  # 第 1 张：Frame 1
ax1.set_title('Frame 1')
ax1.axis('off')

ax2.imshow(frame2, cmap='gray')                                  # 第 2 张：Frame 2
ax2.set_title('Frame 2')
ax2.axis('off')

im = ax3.imshow(mag, cmap='jet')                                 # 第 3 张：光流幅度
ax3.set_title('Optical Flow Magnitude')
ax3.axis('off')

cbar = fig.colorbar(im, cax=cax)                                 # 把 colorbar 画到最右侧
cbar.set_label('Motion Magnitude (pixels/frame)', fontsize=10)

plt.show()

**这次光流结果整体成功，但中间行人的运动被弱化了，需要和原图对照才能解读。**

## 逐图解读

**Frame 1 / Frame 2**

两张原图，画面里有**3 个行人在走动**：

- **左侧行人**：在下坡路上，两帧间位置变化明显。
- **中间行人**：在上方路口附近，两帧间位置变化**较小**。
- **右侧行人**：在右侧路上，两帧间位置变化明显。

**Optical Flow Magnitude（光流幅度图）**

用 jet 色图 + colorbar 显示每个像素的运动幅度：

| 颜色    | 幅度（像素/帧） | 含义             |
| ------- | --------------- | ---------------- |
| 深蓝    | 0 ~ 1           | 几乎不动（背景） |
| 青      | 2 ~ 4           | 运动小           |
| 绿 / 黄 | 4 ~ 7           | 运动中           |
| 橙 / 红 | 8 ~ 10          | 运动剧烈         |

## 三块亮斑对应三个人

**左侧亮斑（青-黄-橙红）**

对应 **Frame 1/2 左侧行人**。他两帧间移动明显，光流值中等偏高，显示为青色到黄橙色。**检测到了**。

**中间亮斑（青蓝色，弱）**

对应 **Frame 1/2 中间行人**。他两帧间移动**很小**，光流值低，只显示为**青蓝色**。虽然检测到了，但**不够醒目**。

**右侧亮斑（青-黄-橙红）**

对应 **Frame 1/2 右侧行人**。他移动明显，光流值高，显示为**黄橙色**。**检测到了**。

## 结果好不好

**好的地方：**

- ✅ **3 个行人全部检测到**（都出现亮斑）。
- ✅ **人形是实心的**，不像帧间差分那样空心。
- ✅ **背景干净**，几乎全是深蓝，没有杂点。
- ✅ **能看出运动强度差异**：左侧和右侧行人亮（运动大），中间行人暗（运动小）。

**不足的地方：**

- ⚠️ **中间行人的运动被弱化**：因为他走得慢，光流值小，显示为青蓝色，不显眼。
- ⚠️ **亮斑形状是"团状"**，不是精确人形。光流反映的是"运动区域"，比人本身大一圈。

## 和帧间差分法对比

同样这段视频，两种方法的结果差别很大：

|          | 帧间差分法    | 密集光流法（本次）     |
| -------- | ------------- | ---------------------- |
| 人形     | 空心轮廓      | **实心亮斑**           |
| 背景噪声 | 有零星白点    | **几乎全蓝，干净**     |
| 运动强度 | 只有"变/没变" | **颜色区分强度**       |
| 方向     | 不知道        | 能（本图只显示幅度）   |
| 慢速目标 | 可能漏检      | **能检测（只是值小）** |

**光流结果明显更干净、更完整。**

## 颜色条（colorbar）的作用

右侧颜色条标注了 **"Motion Magnitude (pixels/frame)"**：

- 数字单位是 **"每帧移动的像素数"**。
- **红色 10** ≈ 每帧移动约 10 像素（运动剧烈）。
- **深蓝 0** ≈ 几乎不动。
- **中间颜色**按数值渐变。

**有了 colorbar，颜色才可读**——否则只看红黄蓝，不知道"红是多快"。

## 一句话

**这次光流检测成功：3 个行人都被检测到，人形实心、背景干净、还能看出运动强度差异。** 唯一不足是中间行人走得慢、亮斑偏弱。整体上**比帧间差分法干净、完整得多**，代价是计算量大，且输出是"运动强度图"，不是精确目标掩码。

# 14. 小结

- 固定阈值：`cv2.threshold()`，五种类型。
- OTSU：自动阈值，适用于双峰直方图，通过最大化类间方差找到最佳阈值。
- 多阈值：通过组合多个二值化结果实现，将图像分为多个亮度区域。
- 自适应阈值：`cv2.adaptiveThreshold()`，为每个像素用邻域均值或高斯加权和减去常数 C 得到局部阈值，适用于光照不均。
- 自适应阈值处理的目的不是让图像“变好看”，而是把图像从“亮度连续场”转换成“区域离散标签”，从而让计算机能够计数、测量、检测和决策。
- 移动平均阈值：通过局部均值实现，计算快、实现简单、可解释性强、自然处理光照渐变、与后续形态学/连通组件兼容好；代价是对噪声不够鲁棒。
- HSV 颜色分割：`cv2.inRange()`，推荐用 `portraitGrass.png` 演示，红色上衣和绿色草地在 H 轴上距离远，分割效果直观。
- 区域生长：`cv2.floodFill()`。
- 区域分离与聚合：连通组件。
- 超像素：`cv2.ximgproc`，属于过分割，只做粗分块，不知道什么是物体；是“预分割”，提供规整的小单元。
- 分水岭：`cv2.watershed()`，属于精分割，必须用 marker 引导，擅长在种子间找精确边界、分开粘连物体。
- 图像分割的层次：超像素负责粗分块、降维、去噪，分水岭负责在种子间找精确边界；两者互补，共同把图像从“像素集合”变成“对象集合”（详见 10.1.8）。
- GrabCut：`cv2.grabCut()`。
- 均值漂移：`cv2.pyrMeanShiftFiltering()`。
- 运动分割：帧间差分、背景差分、光流。